# D36: FULL 90 câu bằng **ReAct + Calculator tool** (Qwen3-4B)

Dạng D36: $|z|^2=K|z+\overline{z}|+C$ và $|z-A+Bi|=|z-C_2+Di|$, đếm số
số phức $z$ thỏa mãn. Phương pháp: đại số hóa $z=x+yi$, tách dấu $|x|$
thành 2 nửa đường tròn, đường thứ hai cho 1 đường thẳng, thế đường thẳng
vào từng nửa, giải ra $y$ (có thể rỗng), suy $x$, đối chiếu dấu ĐÚNG
nhánh, cộng lại. Bản chạy đầy đủ của phương pháp đã kiểm chứng độc lập
với **toàn bộ 91 câu D36** (khớp 91/91) và người dùng đã tự test tay 10
câu (đa dạng cả 3 loại số + cả 3 dạng số nghiệm của 2 nhánh) trong
`KLTN_D36_ReAct_Calculator_1cau.ipynb`.

Model **không tự tính tay bất kỳ phép nào**: mọi phép tính đều gọi tool
`Calculator` (sympy, chính xác tuyệt đối, **có bộ nhớ biến**) theo vòng
lặp ReAct thật:
`Thought → Action → Action Input → Observation → …`

Prompt được **ép mở đầu bằng `<think>\nThought:`** (forced prefix);
model không còn quyền tự chọn viết văn xuôi mở đầu trước khi vào định dạng
ReAct.

Vòng lặp cũng có **chốt chặn "Final Answer"**: nếu model đã viết xong
`Final Answer: \boxed{{<Letter>}}` mà không dừng sinh token sạch (một số
lần nó viết tiếp, lặp lại giải đề từ đầu), harness cắt và dừng ngay tại
đó thay vì chạy tiếp toàn bộ lại lần 2.

## Điểm khác bản 1 câu: vòng lặp ReAct chạy THEO LÔ

Chạy tuần tự 90 câu sẽ mất hàng giờ. Ở đây mỗi **vòng** gọi vLLM **một lần**
cho tất cả các câu đang hoạt động (vLLM tự batching), rồi chạy Calculator
riêng cho từng câu, rồi generate tiếp.

Mỗi câu có **bộ nhớ biến riêng** (`MayTinh()` riêng), **ngân sách token
riêng**, và tự thoát khỏi lô khi viết xong `Final Answer`.

Hạn mức tool gọi/câu (`36          # quy trinh day du ~20 luot tinh (k,c,solve-duong,x_expr,count=0,solve-TH1(list),4x(x-index+check[+count++]),solve-TH2(list),4x(x-index+check[+count++])) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap`) đủ dư cho khoảng 19 lượt tính
đầy đủ quy trình cộng tối đa 4 lượt so khớp phương án.

## Prompt & backend giống hệt bản 1 câu

Cell 3 (prompt + backend `MayTinh`, đặt trước khi `LLM(...)` khởi tạo để
an toàn với `multiprocessing.fork`) được **trích nguyên văn** từ notebook
1 câu bằng script `scratch/build_react_full_D36.py`; không gõ lại, nên
không có nguy cơ lệch giữa 2 bản.

## Trước khi chạy

Upload `plan_solve_prompts_D36.json` (90 câu, đã sinh sẵn từ
`Sinh_them_cau_hoi/So_phuc_day_du.csv`, đã loại câu gốc STT 14 nằm trong
few-shot) thành Kaggle Dataset (slug gợi ý `d36-full90`), gắn vào notebook,
bật GPU.

Kết quả: `/kaggle/working/d36_react_calculator_full90.csv`

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

DATA_PATH = '/kaggle/input/d36-full90/plan_solve_prompts_D36.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d36_react_calculator_full90.csv'

MODEL = 'Qwen/Qwen3-4B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'x', 'y'}  # x,y: phan thuc/ao cua z=x+yi

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                ket_qua_bool = sp.true if bang_nhau else sp.false
                if ten:
                    self.ns[ten] = ket_qua_bool
                    return f'{ten} = {ket_qua_bool}', None
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D36_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai PART_HUONGGIAI va PART_FEWSHOT,
# giu nguyen PART_TOOL va PART_KIENTHUC.

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why). Just like any other computation, `Eq(...)` can be given a name too, e.g. `test_A = Eq(left, right)`; the `True`/`False` result is then genuinely stored under that name and can be referred to again later by that name, exactly like a numeric result.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Every equation in this problem type is at most quadratic (in $y$, after substituting the line), so `solve(...)` always returns an exact closed form; a numeric-only fallback is never needed here.

**NEVER solve a system of equations all at once.** Calling `solve([eq1, eq2], [x, y])` with a LIST of equations and a LIST of unknowns is unreliable: depending on the exact expressions involved, it can return a dictionary instead of a list, and this Calculator cannot handle a dictionary result (it will error). Instead, always solve ONE equation for ONE unknown at a time: solve the first equation for one unknown (giving a list with one expression, possibly in terms of the other unknown), substitute that expression into the second equation, then solve THAT for the remaining unknown. This always returns a plain list, exactly like every other use of `solve(...)` already covered above.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Turning both conditions into plain equations in $x,y$.** Write $z=x+yi$ with $x,y$ real (fact 1). By fact 22, $z+\overline{z}=2x$, and by fact 4, $|z|^2=x^2+y^2$. So the first condition $|z|^2=K|z+\overline{z}|+C$ becomes $x^2+y^2=K|2x|+C=2K|x|+C$: an equation with no $z$ or $\overline{z}$ left in it, just $x,y$, except for that one absolute value around $x$.

**Splitting on the sign of $x$: an absolute value always forces a case split.** $|x|=x$ when $x\ge0$ and $|x|=-x$ when $x<0$, so the single equation $x^2+y^2=2K|x|+C$ is really two DIFFERENT equations depending on which half-plane $x$ falls in:
* If $x\ge0$: $x^2-2Kx+y^2-C=0$.
* If $x<0$: $x^2+2Kx+y^2-C=0$.
Each is the equation of a circle (centered at $(K,0)$ and $(-K,0)$ respectively); but only the half of that circle actually lying in the matching half-plane counts; a point computed from the wrong branch's formula must be checked against ITS OWN sign condition before being trusted, never assumed.

**The second condition, read directly: no abstraction needed.** A condition like $|z-A+Bi|=|z-C+Di|$ (for whatever real numbers $A,B,C,D$ are shown, with whatever signs) turns into a plain equation the moment you substitute $z=x+yi$: $z-A+Bi=(x-A)+(y+B)i$ directly; read each sign exactly as displayed, there is no need to match it against any template or flip anything. Square both sides using the real/imaginary parts you just built: $(x-A)^2+(y+B)^2=(x-C)^2+(y+D)^2$. By fact 11, both sides contain the cluster $x^2+y^2$; solving this single equation for $x$ (in terms of $y$) eliminates that cluster automatically and gives a line $x=\dots$. (If a problem ever showed two points with the same $x$-coordinate, this would fail to solve for $x$; fact 19 explains that case: solve for $y$ in terms of $x$ instead. This does not happen in this problem family, but it is worth knowing why.)

**Why both branches always have to be worked out in full, no matter what the first one gives.** The split in the previous paragraph came from $x\ge0$ and $x<0$; together these cover every real number exactly once, with no overlap and no gap between them. That is what makes the two branches genuinely INDEPENDENT problems rather than two attempts at the same problem: the $x\ge0$ branch's equation only ever describes points with $x\ge0$, and the $x<0$ branch's equation only ever describes points with $x<0$, so nothing about how many valid points turn up in one branch can say anything about how many turn up in the other; they were built from different substitutions ($|x|=x$ versus $|x|=-x$) applied to a completely different half of the plane. Finding two valid candidates in the $x\ge0$ branch does not make the $x<0$ branch less likely to also have valid candidates, and finding zero valid candidates in the $x\ge0$ branch does not make the $x<0$ branch more likely to have some either; each branch's count of valid candidates depends only on where THAT branch's own circle happens to cross the line, a completely separate geometric question. So solving and checking one branch is never a substitute for solving and checking the other; both must always be carried through to completion before the final count is known.

**Feeding the line into each half separately.** Substitute the line's expression for $x$ into the FIRST branch's circle equation ($x\ge0$'s version) and solve for $y$: this is now a single-unknown quadratic. Solving it can give TWO real roots, or it can give NONE (an empty list) when the line simply does not cross that particular half of the circle at all; both outcomes are normal and expected, not an error; an empty list just means this branch contributes 0 candidates, and you move directly to the other branch without trying to invent a root that is not there. Each root $y$ that IS returned gives a candidate point: recover $x$ from the line's formula. Do the exact same substitution into the SECOND branch's circle equation ($x<0$'s version) to get a second, independent list of $y$ roots (again possibly empty) and candidate points.

**Every candidate must be checked against its OWN branch's sign condition: never assumed.** A point coming from the $x\ge0$ branch's equation is only a genuine solution if its actual computed $x$ is $\ge0$; a point from the $x<0$ branch's equation is only genuine if its computed $x$ is $<0$ (note: exactly $x=0$ fails this second check, even though it passes the first; a point landing exactly on the boundary belongs only to the $x\ge0$ branch, never to the $x<0$ branch). Check each of the (up to 4) candidates individually; do not assume a branch's roots are automatically valid just because you solved its equation correctly; solving the equation only finds points ON that branch's full circle, not necessarily on the correct half of it.

**Totalling.** The final count is exactly the number of candidates (across both branches) that passed their own branch's sign check; simply count them; no separate overlap check is needed, since a point cannot simultaneously satisfy $x\ge0$ and $x<0$.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Có bao nhiêu số phức $z$ thỏa mãn $|z|^2 = 2|z+\overline{z}|+4$ và $|z-1-i| = |z-3+3i|$?
A. $4$.   B. $3$.   C. $1$.   D. $2$.

<think>
Thought: First condition, quoted verbatim: "$|z|^2=2|z+\overline{z}|+4$"; the coefficient is $2$, so $k=2$; the constant is $+4$, so $c=4$. I store them.
Action: Calculator
Action Input: k = 2
Observation: k = 2

Thought: Now read off $c$, the constant term.
Action: Calculator
Action Input: c = 4
Observation: c = 4

Thought: Second condition, quoted verbatim: "$|z-1-i|=|z-3+3i|$". Reading each sign exactly as shown: $z-1-i=(x-1)+(y-1)i$, and $z-3+3i=(x-3)+(y+3)i$. Squaring both sides and setting them equal gives the line's equation; solve it for $x$.
Action: Calculator
Action Input: solve(Eq((x-1)**2+(y-1)**2, (x-3)**2+(y+3)**2), x)
Observation: [2*y + 4]

Thought: So $x=2y+4$ on this line. Store it.
Action: Calculator
Action Input: x_expr = 2*y+4
Observation: x_expr = 2*y + 4

Thought: Before checking any candidate, start a running count at 0; I will update this variable after every single check instead of adding things up later from memory.
Action: Calculator
Action Input: count = 0
Observation: count = 0

Thought: Substitute into the $x\ge0$ branch's circle equation and solve for $y$. I store the whole list under a name so I can index into it afterward, instead of retyping any root by hand.
Action: Calculator
Action Input: roots1 = solve(Eq(x_expr**2 - 2*k*x_expr + y**2, c), y)
Observation: roots1 = [-2, 2/5]

Thought: This list has 2 roots, so branch 1 has exactly 2 candidates to check: `roots1[0]` and `roots1[1]`. Starting with `roots1[0]`; recover its $x$ directly by index, never by retyping the value.
Action: Calculator
Action Input: x_a = x_expr.subs(y, roots1[0])
Observation: x_a = 0

Thought: Check the $x\ge0$ condition for this candidate.
Action: Calculator
Action Input: x_a >= 0
Observation: True

Thought: Observation is True, so this candidate is valid; I now call `count = count + 1`.
Action: Calculator
Action Input: count = count + 1
Observation: count = 1

Thought: Now the second candidate from this branch: `roots1[1]`.
Action: Calculator
Action Input: x_b = x_expr.subs(y, roots1[1])
Observation: x_b = 24/5

Thought: Check the $x\ge0$ condition.
Action: Calculator
Action Input: x_b >= 0
Observation: True

Thought: Observation is True, so this candidate is valid; I now call `count = count + 1`.
Action: Calculator
Action Input: count = count + 1
Observation: count = 2

Thought: Branch 1's list had 2 roots, and both are now checked ($x_a$ valid, $x_b$ valid); branch 1 is complete. Moving to branch 2 next, which is mandatory regardless of what branch 1 found: substitute the line into its circle equation and solve for $y$, again storing the whole list under a name.
Action: Calculator
Action Input: roots2 = solve(Eq(x_expr**2 + 2*k*x_expr + y**2, c), y)
Observation: roots2 = [-14/5, -2]

Thought: This list has 2 roots, so branch 2 has exactly 2 candidates to check: `roots2[0]` and `roots2[1]`. Starting with `roots2[0]`.
Action: Calculator
Action Input: x_c = x_expr.subs(y, roots2[0])
Observation: x_c = -8/5

Thought: Check the $x<0$ condition for this candidate.
Action: Calculator
Action Input: x_c < 0
Observation: True

Thought: Observation is True, so this candidate is valid; I now call `count = count + 1`.
Action: Calculator
Action Input: count = count + 1
Observation: count = 3

Thought: Now the second candidate from this branch: `roots2[1]`.
Action: Calculator
Action Input: x_d = x_expr.subs(y, roots2[1])
Observation: x_d = 0

Thought: Check the $x<0$ condition for this candidate; note this is the SAME point as $x_a$, but here it must satisfy $x<0$, not $x\ge0$.
Action: Calculator
Action Input: x_d < 0
Observation: False

Thought: Observation is False, so this candidate is not valid; I do NOT call `count = count + 1` for it. Branch 2's list had 2 roots, and both are now checked ($x_c$ valid, $x_d$ invalid); branch 2 is complete. Branch 1's 2 roots were also both actually checked earlier ($x_a$, $x_b$, with real Action calls, not assumed). With both branches' real computations done, `count` now holds the final total; I query it to confirm.
Action: Calculator
Action Input: count
Observation: 3

Thought: `count` is confirmed to be $3$. Before testing anything, I read all four options verbatim from the problem text, in order A, B, C, D, and store them together in a single list, in that same order. This is the only time I read these values from the problem text; from now on I refer to each one only by index, never by retyping it.
Action: Calculator
Action Input: opts = [4, 3, 1, 2]
Observation: opts = [4, 3, 1, 2]

Thought: Now I go through the four options one at a time, carefully, in order, comparing `count` against the matching element of `opts` by index; I test every single one, even after a match is found, since `Eq()` here is a numeric check, not a formal proof. Starting with option A, which is `opts[0]`. Test it.
Action: Calculator
Action Input: Eq(count, opts[0])
Observation: False

Thought: A did not match. Option B is `opts[1]`. Test it.
Action: Calculator
Action Input: Eq(count, opts[1])
Observation: True

Thought: B matched - but I still do not stop, since I have not yet tested every option. Option C is `opts[2]`. Test it.
Action: Calculator
Action Input: Eq(count, opts[2])
Observation: False

Thought: C did not match. Option D is `opts[3]`. Test it.
Action: Calculator
Action Input: Eq(count, opts[3])
Observation: False

Thought: All four options are now tested. Exactly one came back True: option B (`opts[1]`). That is the answer.
</think>

Final Answer: \boxed{B}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always has the form: how many complex numbers $z$ satisfy $|z|^2=K|z+\overline{z}|+C$ (for a given coefficient $K$ and constant $C$, $C$ possibly negative) AND $|z-A+Bi|=|z-C_2+D i|$ (for given real numbers, whatever signs are shown)? Follow these steps, using the Calculator for every computation.

1. **Finding $k$ and $c$.** Quote the first condition verbatim, then read off the coefficient of $|z+\overline{z}|$ as $k$ (if none is written, $k=1$), and the constant term as $c$, keeping whatever sign is shown. Store both.
2. **Building the line.** Quote the second condition verbatim. Reading each sign exactly as displayed (no flipping, no template-matching; substitute $z=x+yi$ directly into each side), write the two sides as $(x\pm A)^2+(y\pm B)^2$ and $(x\pm C_2)^2+(y\pm D)^2$. Solve `Eq(...)` of these two squared expressions for $x$ to get the line as `x_expr` (a formula in $y$).
3. **Start a running count.** Store `count = 0`. Every time, and only when, a candidate passes its branch's sign check below, update it immediately with `count = count + 1`; never sum several results later from memory.
4. **How to process ONE branch.** Substitute `x_expr` into that branch's circle equation and solve for $y$, storing the WHOLE list under a name (e.g. `roots1 = solve(...)`), even if it has 0, 1, or 2 roots (an empty list is not an error; it simply means this branch contributes nothing). In the Thought immediately before your next Action (never a standalone Thought with no Action of its own), state how many roots the list has. For EACH root, in order, indexed (`roots1[0]`, then `roots1[1]` if it exists; never retyped): compute the matching $x$, check the branch's own sign condition, and update `count` right away if it passes. Once every root is checked, fold a one-line confirmation into the Thought that introduces your NEXT Action (no extra Action just for this), e.g. "branch 1's 2 roots are all checked, 1 valid".
5. **Branch 1 ($x\ge0$).** Apply step 4 to $x^2-2kx+y^2=c$, checking `x >= 0`. Once branch 1's confirmation is stated, your very next Action, with nothing else in between, is branch 2's first computation, step 6's `roots2 = solve(...)`. Branch 1 finishing is never, by itself, a sign that branch 2 is also done, or that `count` is already final; branch 2 has not been computed at all yet, and skipping straight to querying `count` here would leave branch 2 entirely unchecked.
6. **Branch 2 ($x<0$): MANDATORY, regardless of what branch 1 found.** Apply step 4 to $x^2+2kx+y^2=c$, checking `x < 0` (a computed $x$ of exactly $0$ fails this, even if it passed branch 1's check for a different root). PART 2 explains why the two branches never predict each other's results: finishing branch 1, with any outcome at all, is never a reason to skip or shortcut branch 2.
7. Once both branches' closing confirmations (from step 4, applied once per branch) have been stated, `count` holds the final total; there is no separate summing step. Your very next Action, with nothing else in between and no Final Answer yet, is step 8: storing the four options. Reaching a final `count` is never, by itself, a reason to stop; the options have not been read or tested yet.
8. **Storing all four options before testing any of them.** Read all four options' values VERBATIM from the problem text, in order A, B, C, D, and store them together in ONE Calculator call as a single list: `opts = [<A>, <B>, <C>, <D>]`. This is the only time you ever read these values from the problem text; from here on, refer to each one only by index (`opts[0]` for A, `opts[1]` for B, `opts[2]` for C, `opts[3]` for D), never by retyping a number.
9. For each option A, B, C, D in turn: compare with `Eq(count, opts[<matching index>])`. Test every option even after one already returns `True`; `Eq()` here is a numeric check, not a formal proof.
10. Write the final line `Final Answer: \boxed{<Letter>}` using whichever option matched.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 36          # quy trinh day du ~20 luot tinh (k,c,solve-duong,x_expr,count=0,solve-TH1(list),4x(x-index+check[+count++]),solve-TH2(list),4x(x-index+check[+count++])) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')
FINAL_ANSWER_RE = re.compile(r'Final\s+Answer\s*:\s*\\boxed\{\s*[A-D]\s*\}')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        # CHOT CUNG: mot so lan model viet xong Final Answer nhung khong
        # dung sinh sach (khong phat EOS), roi viet tiep 1 vong giai lai
        # tu dau, ket thuc vong do bang "Observation:" moi khien harness
        # tuong la 1 luot goi tool that va lap tiep. Cat ngay tai cho
        # Final Answer XUAT HIEN LAN DAU, bat ke stop_reason la gi.
        m_final = FINAL_ANSWER_RE.search(s.full_text)
        if m_final:
            s.full_text = s.full_text[:m_final.end()]
            s.xong = True
            s.ly_do_dung = 'model_ket_thuc'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)